# 02 — Feature Engineering
Imputation · Encoding · Derived features · Log-transform target

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/train.csv')
print(f'Loaded: {df.shape}')

## Derived Features

In [ ]:
df['TotalSF']   = df['TotalBsmtSF'] + df['1stFlrSF'] + df['2ndFlrSF']
df['HouseAge']  = df['YrSold'] - df['YearBuilt']
df['RemodAge']  = df['YrSold'] - df['YearRemodAdd']
df['TotalBath'] = (df['FullBath'] + 0.5 * df['HalfBath']
                   + df['BsmtFullBath'] + 0.5 * df['BsmtHalfBath'])
df['HasGarage'] = (df['GarageArea'] > 0).astype(int)
df['HasPool']   = (df['PoolArea']   > 0).astype(int)

print('Derived features added:')
print(df[['TotalSF', 'HouseAge', 'RemodAge', 'TotalBath', 'HasGarage', 'HasPool']].describe())

## Log-transform Target

In [ ]:
df['SalePrice_log'] = np.log1p(df['SalePrice'])
print(f'SalePrice skew (raw)     : {df["SalePrice"].skew():.3f}')
print(f'SalePrice skew (log1p)   : {df["SalePrice_log"].skew():.3f}')

## Define Feature Sets

In [ ]:
# Ordinal features — have a meaningful order
ORDINAL_FEATURES = {
    'ExterQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'ExterCond': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtQual':  ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'BsmtCond':  ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'HeatingQC': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'KitchenQual': ['Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'FireplaceQu': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageQual': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'GarageCond': ['NA', 'Po', 'Fa', 'TA', 'Gd', 'Ex'],
    'PoolQC':    ['NA', 'Fa', 'TA', 'Gd', 'Ex'],
}

# Nominal features — no inherent order
NOMINAL_FEATURES = [
    'MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour',
    'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood',
    'Condition1', 'Condition2', 'BldgType', 'HouseStyle',
    'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd',
    'MasVnrType', 'Foundation', 'BsmtExposure', 'BsmtFinType1',
    'BsmtFinType2', 'Heating', 'CentralAir', 'Electrical',
    'Functional', 'GarageType', 'GarageFinish', 'PavedDrive',
    'Fence', 'MiscFeature', 'SaleType', 'SaleCondition',
]

# Numeric features (including derived)
NUMERIC_FEATURES = [
    'LotFrontage', 'LotArea', 'MasVnrArea', 'BsmtFinSF1',
    'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF',
    '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath',
    'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr',
    'KitchenAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 'GarageYrBlt',
    'GarageCars', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF',
    'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea',
    'MiscVal', 'MoSold', 'YrSold', 'OverallQual', 'OverallCond',
    'TotalSF', 'HouseAge', 'RemodAge', 'TotalBath', 'HasGarage', 'HasPool',
]

TARGET = 'SalePrice_log'

print(f'Ordinal features : {len(ORDINAL_FEATURES)}')
print(f'Nominal features : {len(NOMINAL_FEATURES)}')
print(f'Numeric features : {len(NUMERIC_FEATURES)}')

## Train / Test Split

In [ ]:
all_features = NUMERIC_FEATURES + list(ORDINAL_FEATURES.keys()) + NOMINAL_FEATURES
# Keep only columns present in dataframe
all_features = [c for c in all_features if c in df.columns]

X = df[all_features]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}  |  Test: {X_test.shape}')

# Save for subsequent notebooks
import pickle, os
os.makedirs('../data', exist_ok=True)
with open('../data/splits.pkl', 'wb') as f:
    pickle.dump((X_train, X_test, y_train, y_test,
                 NUMERIC_FEATURES, ORDINAL_FEATURES, NOMINAL_FEATURES), f)
print('Splits saved to data/splits.pkl')